# Road Damage Inspection System

<p style="color:#526173"><strong>Austin Wang</strong> · 0:00–1:00</p>

## 8:17 a.m. — one road image, three urgent questions

> **What broke? Where is it? How much road surface is affected?**

<p align="center"><img src="artifacts/final/marked_pothole_hook.jpg" alt="A road pothole circled in white paint for repair" width="920"></p>

**Opening line:** An inspection vehicle may see this road only once, but a city may have thousands of frames waiting. Our goal is to turn each frame into visible evidence for an inspector—not an autonomous repair order.

**Project in one sentence:** road pixels → boxes + masks → explained priority → human judgment.


# One image, two specialists, one reviewable decision

<p style="color:#526173"><strong>Austin Wang</strong> · 1:00–2:00</p>

<p align="center"><img src="artifacts/presentation/project_workflow.svg" alt="End-to-end project workflow from data and road image to two models, evidence fusion, and human review" width="1100"></p>

**Takeaway:** YOLO answers **what and where**; SegFormer answers **how much area**. Their outputs are fused only to explain and rank cases for human review.


# What do D00, D10, D20, and D40 mean?

<p style="color:#526173"><strong>Austin Wang</strong> · 2:00–3:00</p>

RDD2022 uses four detection codes, each paired with a bounding box:

| Code | Meaning | Visual cue |
|---|---|---|
| **D00** | Longitudinal crack | Runs along the road |
| **D10** | Transverse crack | Crosses the road |
| **D20** | Alligator crack | Connected web of cracks |
| **D40** | Pothole | Local broken/depressed pavement |

<p align="center"><img src="artifacts/member1/EDA_Figures/09_damage_class_vocabulary.png" alt="RDD2022 examples of D00, D10, D20, and D40 with bounding boxes" width="960"></p>

**Takeaway:** the codes identify **damage type, not severity**—D40 is not automatically “level 40” damage.


# Two public datasets, two complementary label types

<p style="color:#526173"><strong>Austin Wang</strong> · 3:00–4:00</p>

| Data source | What it contains | Labels | Role in our project |
|---|---:|---|---|
| **RDD2022** | 47,420 images; 38,385 publicly labeled | 55,006 valid target boxes across six countries / seven capture domains | Four-class object detection |
| **Pothole Mix** | 4,340 image-mask pairs from six component sources | Pixel masks; official 3,340 / 496 / 504 split | Binary pothole segmentation |

<p align="center"><img src="artifacts/member3/eda/m3_fig5_aligned_samples.png" alt="Aligned Pothole Mix road images and segmentation masks" width="900"></p>

**Takeaway:** these datasets are related road imagery but solve different cognitive problems: RDD2022 supplies **boxes and types**, while Pothole Mix supplies **pixel-level pothole shape**.


# Member 1: make the data trustworthy before modeling

<p style="color:#526173"><strong>Austin Wang</strong> · 4:00–5:00</p>

| Full-data audit | Verified result | Decision |
|---|---:|---|
| Official RDD2022 images decoded | 47,420 / 47,420 | No corrupt image removal |
| Publicly labeled images retained | 38,385 | Use the complete labeled pool |
| Raw XML boxes / target boxes | 65,712 / 55,007 | Audit 10,705 non-target codes; do not remap them |
| Exportable target boxes | 55,006 | Exclude one degenerate D20 box |
| Exact / exact+near duplicate rows | 4 / 3,256 | Retain but group-lock to one split |
| Valid target negatives | 14,618 images | Retain to teach background |

**Method chain:** Pascal VOC XML → image/box tables → integrity and duplicate audit → group-aware split → synchronized YOLO + COCO exports.

**Takeaway:** Member 1’s main product is not just EDA—it is a fixed, leakage-aware data contract shared by both detectors.


# EDA prediction: small and rare damage will be the hard case

<p style="color:#526173"><strong>Austin Wang</strong> · 5:00–6:00</p>

<table><tr>
<td width="50%"><img src="artifacts/member1/EDA_Figures/01_counts_country_and_class.png" alt="Image counts by country and target boxes by class" width="100%"></td>
<td width="50%"><img src="artifacts/member1/EDA_Figures/04_box_area_violin_and_center_heatmap.png" alt="Object scale by class and object center heatmap" width="100%"></td>
</tr></table>

- Japan contributes 10,506 labeled images versus 2,829 from Czech.
- D00 supplies 47.3% of target boxes; D40 supplies only 11.9%.
- The median target occupies 1.43% of the image; D40 has the smallest class median (0.54%).

**Takeaway:** imbalance and tiny targets mean that overall mAP is insufficient—we need per-class recall, especially for potholes.


# From EDA to a fair evaluation design

<p style="color:#526173"><strong>Austin Wang</strong> · 6:00–6:30</p>

<p align="center"><img src="artifacts/member1/EDA_Figures/05_resolution_brightness_contrast_blur.png" alt="Resolution, brightness, contrast, and blur distributions by country" width="930"></p>

| Leakage-safe split | Images |
|---|---:|
| Train | 26,888 |
| Validation | 5,714 |
| Test | 5,783 |
| Held-out-US auxiliary test | 4,805 |

Norway’s median resolution is 8.22 MP while most domains are roughly 0.26–0.52 MP; brightness and capture viewpoint also vary by country. Duplicate groups never cross the primary split.

**Takeaway:** a random held-out test measures average performance; the separate held-out-US test asks whether the detector travels to a new geography.


# Member 2: a controlled champion–challenger experiment

<p style="color:#526173"><strong>Kevin Fan</strong> · 6:30–7:30</p>

| Controlled factor | YOLO11n | YOLO11s |
|---|---:|---:|
| Role | Lightweight baseline | Capacity challenger |
| Parameters | 2.58 M | 9.41 M |
| Training manifest | Same hash-pinned 8,000 images | Same hash-pinned 8,000 images |
| Validation / test | Full 5,714 / 5,783 | Full 5,714 / 5,783 |
| Schedule | 30 epochs, 640 px, same seed/augmentation | Identical |

**Metric guide:** mAP@0.50 asks whether damage was found with reasonable overlap; mAP@0.50:0.95 rewards tight localization; recall measures missed damage; F1 balances precision and recall at a validation-selected confidence.

**Takeaway:** the comparison isolates model capacity—data composition, evaluation set, threshold-selection rule and timing hardware remain controlled.


# Detection result: the larger model wins where it matters

<p style="color:#526173"><strong>Kevin Fan</strong> · 7:30–9:00</p>

<p align="center"><img src="artifacts/presentation/detection_model_comparison.png" alt="YOLO11n versus YOLO11s held-out detection metrics and compute tradeoff" width="980"></p>

| Shared test metric | YOLO11n | YOLO11s |
|---|---:|---:|
| mAP@0.50 | 0.4336 | **0.4439** |
| mAP@0.50:0.95 | 0.2033 | **0.2080** |
| Recall / F1 | 0.4268 / 0.4627 | **0.4447 / 0.4783** |
| D40 AP / recall | 0.2865 / 0.2621 | **0.3209 / 0.3024** |
| Single-image latency (L4) | 16.07 ms | 16.28 ms |

**Takeaway:** YOLO11s is the detection champion because it improves both overall detection and pothole recall with effectively unchanged single-image latency.


# The champion gain is real—but training was compute-limited

<p style="color:#526173"><strong>Kevin Fan</strong> · 9:00–10:00</p>

<p align="center"><img src="artifacts/presentation/detection_learning_curves.png" alt="YOLO11n and YOLO11s learning curves across 30 epochs" width="900"></p>

| Paired bootstrap micro-F1 | YOLO11n | YOLO11s | Delta (95% CI) |
|---|---:|---:|---:|
| Overall | 0.4636 | 0.4830 | **+0.0194** [+0.0112, +0.0268] |
| D40 | 0.3215 | 0.3630 | **+0.0416** [+0.0158, +0.0688] |

Both primary runs achieve their best validation score at epoch 30.

**Takeaway:** the YOLO11s gain is statistically distinguishable from zero, but both curves suggest the compute-limited training schedule had not fully saturated.


# Generalization test: unseen geography costs about one sixth of mAP

<p style="color:#526173"><strong>Kevin Fan</strong> · 10:00–11:15</p>

| Training geography | Strict US slice | Matched non-US slice |
|---|---:|---:|
| All-country 8k | **0.5064** | 0.4247 |
| Non-US 8k | 0.4206 | **0.4391** |

<p align="center"><img src="artifacts/member2/runs/figures/B8_miss_rate_slices.png" alt="Detection miss rates by class, object size, country, blur, and brightness" width="900"></p>

- Removing US training data reduces strict US mAP@0.50 by **16.9% relative**.
- Small-tercile boxes are missed **74.0%** of the time even by YOLO11s.
- D40 remains the worst class: **71.7% miss rate** at the selected operating point.

**Takeaway:** object size is the dominant failure driver, and geography adds a second measurable risk.


# Error analysis: localization and missed small damage dominate

<p style="color:#526173"><strong>Kevin Fan</strong> · 11:15–12:30</p>

<div style="display:flex;gap:12px;justify-content:center">
  <img src="artifacts/member2/runs/figures/B8_qualitative_panels.png" alt="Dense multi-damage example with ground truth and detector predictions" style="width:49%;height:350px;object-fit:cover;object-position:50% 0%">
  <img src="artifacts/member2/runs/figures/B8_qualitative_panels.png" alt="Small-damage example with ground truth and detector predictions" style="width:49%;height:350px;object-fit:cover;object-position:50% 31%">
</div>
<p align="center"><small>Selected views from the five-case qualitative panel; the complete panel remains in the master notebook.</small></p>

| False-positive type (YOLO11s) | Share | What it means |
|---|---:|---|
| Localization | 43.8% | Correct damage, loose box |
| Background | 33.1% | Shadows, seams, markings, repairs |
| Duplicate | 18.4% | Extra box on an already found target |
| Wrong class | 4.7% | Taxonomy confusion is uncommon |

**Takeaway:** the bottleneck is not learning the four names; it is seeing tiny damage and drawing one tight box around it.


# Member 3: segmentation begins with a sparse-foreground problem

<p style="color:#526173"><strong>RJ Xia</strong> · 12:30–13:45</p>

<table><tr>
<td width="50%"><img src="artifacts/member3/eda/m3_fig2_foreground_imbalance.png" alt="Pothole foreground imbalance in Pothole Mix" width="100%"></td>
<td width="50%"><img src="artifacts/member3/eda/m3_fig5_aligned_samples.png" alt="Aligned road images and pothole masks" width="100%"></td>
</tr></table>

- All **4,340** image-mask pairs decode correctly; no dimension mismatch.
- Only **1,184 images (27.3%)** contain pothole pixels.
- Among positive images, the median pothole covers only **2.58%** of the frame.
- Green crack pixels are background for this binary pothole task, creating valid hard negatives.

**Takeaway:** pixel accuracy would look strong by predicting mostly road, so model selection must focus on pothole IoU, Dice, recall and boundaries.


# Two segmentation models, one shared evaluation stack

<p style="color:#526173"><strong>RJ Xia</strong> · 13:45–14:45</p>

| | DeepLabV3–MobileNetV3 | SegFormer-B0 |
|---|---|---|
| Core idea | Atrous convolution + multi-scale CNN context | Hierarchical transformer + lightweight MLP decoder |
| Expected strength | Local detail, recall, lower activation memory | Global/multi-scale context, compact parameter count |
| Main risk | Boundary loss and limited long-range context | Upsampling boundaries and lower recall |

**Shared protocol:** official 3,340 / 496 / 504 split, 512×512 input, paired augmentation, cross-entropy + soft-Dice loss, 40 epochs, validation-selected checkpoint and threshold.

**Metric guide:** IoU penalizes both missing and extra mask area; Dice summarizes overlap; recall measures missed pothole pixels; Boundary F1 tests whether the predicted edge follows the true edge.

**Takeaway:** architecture changes, but data, loss, evaluation code and test policy stay fixed.


# Segmentation result: a narrow win with a meaningful tradeoff

<p style="color:#526173"><strong>RJ Xia</strong> · 14:45–16:00</p>

<table><tr>
<td width="50%"><img src="artifacts/member3/evaluation/m3_fig7_training_curves.png" alt="DeepLabV3 and SegFormer training curves" width="100%"></td>
<td width="50%"><img src="artifacts/member3/evaluation/m3_fig8_confusion_matrices.png" alt="Segmentation confusion matrices" width="100%"></td>
</tr></table>

| Test metric | DeepLabV3 | SegFormer-B0 |
|---|---:|---:|
| Pothole IoU / Dice | 0.6533 / 0.7903 | **0.6646 / 0.7985** |
| Precision / recall | 0.7889 / **0.7917** | **0.8623** / 0.7436 |
| Boundary F1 | 0.7188 | **0.7334** |
| Parameters | 11.02 M | **3.71 M** |

**Takeaway:** SegFormer wins overlap, precision and boundary quality with 3× fewer parameters; DeepLabV3 remains the recall-oriented alternative.


# Look beyond the mean: best and worst segmentation cases

<p style="color:#526173"><strong>RJ Xia</strong> · 16:00–17:00</p>

<div style="display:flex;gap:12px;justify-content:center">
  <img src="artifacts/member3/evaluation/m3_fig11_qualitative_panels.png" alt="Strong pothole segmentation cases" style="width:49%;height:360px;object-fit:cover;object-position:50% 0%">
  <img src="artifacts/member3/evaluation/m3_fig11_qualitative_panels.png" alt="Difficult pothole segmentation cases" style="width:49%;height:360px;object-fit:cover;object-position:50% 100%">
</div>
<p align="center"><small>Selected strong and difficult cases; the complete comparison panel remains in the master notebook.</small></p>

Close potholes are segmented well (IoU roughly 0.88–0.92), while thin, shallow, distant damage can be missed completely. The per-image IoU distribution is therefore bimodal: a single mean hides a genuine failure cluster.

**Takeaway:** the champion is not “solved”—qualitative failures explain why boundary metrics and human review remain necessary.


# From two models to one human-review application

<p style="color:#526173"><strong>RJ Xia</strong> · 17:00–18:00</p>

<table><tr>
<td width="50%"><img src="artifacts/member3/app/CPU-2.png" alt="Verified CPU Gradio road damage analysis" width="100%"></td>
<td width="50%"><img src="artifacts/member3/app/GPU.png" alt="Verified GPU Gradio road damage analysis" width="100%"></td>
</tr></table>

The app shows detection boxes, the segmentation overlay, confidences, damaged-area percentage, model/runtime metadata and a transparent Low/Medium/High prototype priority with reasons.

The captured CUDA run completed the full YOLO11s + SegFormer path in **56 ms**; CPU first-call latency exceeded one second. These app timings include end-to-end overhead and are separate from controlled per-model benchmarks.

**Takeaway:** the interface exposes evidence and limitations; its priority is a triage suggestion, not a civil-engineering rating.


# Final scorecard: champion does not mean universally best

<p style="color:#526173"><strong>Austin Wang</strong> · 18:00–18:45</p>

<p align="center"><img src="artifacts/final/model_tradeoff_summary.svg" alt="Detection and segmentation champion-challenger tradeoff summary" width="1100"></p>

**Takeaway:** both champions win narrowly. YOLO11s earns deployment preference through significant pothole-recall gains; SegFormer wins overlap and size, but DeepLabV3 is defensible when missed potholes cost more than false alarms.


# Deployment must be observable, versioned, and reversible

<p style="color:#526173"><strong>Kevin Fan</strong> · 18:45–19:20</p>

<p align="center"><img src="artifacts/final/deployment_architecture.svg" alt="Versioned road damage deployment architecture with monitoring and rollback" width="1080"></p>

**Operational loop:** monitor → label reviewed failures → train a versioned challenger → gate on locked tests → canary → promote or roll back.

**Takeaway:** model operations preserve the same principle as our experiment design: version everything, protect the test set, measure drift and keep a previous champion ready.


# Conclusion: decision support, not autonomous maintenance

<p style="color:#526173"><strong>RJ Xia</strong> · 19:20–20:00</p>

### What remains hard

- **Small-object recall:** YOLO11s still misses 74% of small-tercile boxes; D40 recall is 0.3024.
- **Domain shift:** removing US training data costs 16.9% relative mAP on strict US images.
- **Localization and boundaries:** detection mAP@0.50:0.95 is about 0.21; segmentation Boundary F1 is about 0.73.
- **Coverage:** night, snow, water, glare, unusual cameras and new jurisdictions are not acceptance-tested.

### Final answer to the opening questions

- **What broke and where?** YOLO11s provides four-class boxes.
- **How much area?** SegFormer-B0 provides the pothole mask and footprint.
- **Who decides?** A human inspector, with evidence and explicit uncertainty.

> **Closing line:** Our strongest result is not automation—it is an auditable way to prioritize road images without hiding where the models fail.

# Questions?


# Appendix — exact handoff route and sources (not part of the timed talk)

| Time | Speaker | Slides | Core message |
|---:|---|---|---|
| 0:00–6:30 | Austin Wang | 1–7 | Problem, label vocabulary, data audit, EDA and split design |
| 6:30–12:30 | Kevin Fan | 8–12 | Controlled detection comparison, significance, generalization and errors |
| 12:30–18:00 | RJ Xia | 13–17 | Segmentation data, models, metrics, failures and app |
| 18:00–18:45 | Austin Wang | 18 | Combined champion tradeoffs |
| 18:45–19:20 | Kevin Fan | 19 | Deployment and maintenance |
| 19:20–20:00 | RJ Xia | 20 | Limitations and conclusion |

## Key sources

1. Arya, D. et al. RDD2022 dataset and paper: https://doi.org/10.6084/m9.figshare.21431547 and https://doi.org/10.1002/gdj3.260 (CC BY 4.0).
2. Pothole Mix v1.0: https://doi.org/10.17632/kfth5g2xk3.2; component-license record in `artifacts/member3/pothole_mix_provenance.json`.
3. Ultralytics YOLO11: https://docs.ultralytics.com/models/yolo11.
4. Chen, L.-C. et al., DeepLabV3: https://arxiv.org/abs/1706.05587; Howard, A. et al., MobileNetV3: https://arxiv.org/abs/1905.02244.
5. Xie, E. et al., SegFormer: https://arxiv.org/abs/2105.15203.
6. Gradio: https://www.gradio.app/docs.
7. Opening photograph: Prosthetic Head, “Marked Pothole,” Wikimedia Commons, CC BY-SA 4.0: https://commons.wikimedia.org/wiki/File:Marked_Pothole.jpg.

Exact methods, complete tables, code, saved outputs and the full reference list remain in `Road_Damage_Final_Project_Master_New.ipynb`.
